In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data_dir = '/content/drive/MyDrive/Assignment5/dataset'

In [ ]:
import os

print(os.listdir('/content/drive/MyDrive/Assignment5/dataset'))

['test', 'train', 'val']


In [ ]:
print(os.listdir('/content/drive/MyDrive/Assignment5/dataset/train'))

['duck', 'chicken']


In [ ]:
def count_images(path):
    for split in ['train', 'val', 'test']:
        print(f"\n{split.upper()}:")
        for cls in ['chicken', 'duck']:
            folder = os.path.join(path, split, cls)
            print(f"{cls}: {len(os.listdir(folder))}")

count_images('/content/drive/MyDrive/Assignment5/dataset')


TRAIN:
chicken: 275
duck: 275

VAL:
chicken: 52
duck: 109

TEST:
chicken: 172
duck: 310


In [ ]:
import os
import random

def balance_dataset(path):
    chicken_path = os.path.join(path, 'train', 'chicken')
    duck_path = os.path.join(path, 'train', 'duck')

    chicken_files = os.listdir(chicken_path)
    duck_files = os.listdir(duck_path)

    target_size = len(chicken_files)

    # Randomly remove extra duck images
    duck_to_remove = random.sample(duck_files, len(duck_files) - target_size)

    for file in duck_to_remove:
        os.remove(os.path.join(duck_path, file))

    print("Balanced dataset!")

balance_dataset('/content/drive/MyDrive/Assignment5/dataset')

Balanced dataset!


In [ ]:
import os

def count_images(path):
    for split in ['train', 'val', 'test']:
        print(f"\n{split.upper()}:")
        for cls in ['chicken', 'duck']:
            folder = os.path.join(path, split, cls)
            print(f"{cls}: {len(os.listdir(folder))}")

count_images('/content/drive/MyDrive/Assignment5/dataset')


TRAIN:
chicken: 275
duck: 275

VAL:
chicken: 52
duck: 109

TEST:
chicken: 172
duck: 310


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from sklearn.metrics import classification_report

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

data_dir = '/content/drive/MyDrive/Assignment5/dataset'

Using device: cuda


In [ ]:
# Transforms
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(20),
        transforms.ToTensor(),
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ]),
}

In [ ]:
# Load data
image_datasets = {
    x: datasets.ImageFolder(f"{data_dir}/{x}", transform=data_transforms[x])
    for x in ['train', 'val']
}

dataloaders = {
    x: torch.utils.data.DataLoader(image_datasets[x], batch_size=16, shuffle=True)
    for x in ['train', 'val']
}

class_names = image_datasets['train'].classes
print("Classes:", class_names)

Classes: ['chicken', 'duck']


In [ ]:
# Model
model = models.resnet18(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 133MB/s]


In [ ]:
# Freeze layers
for param in model.parameters():
    param.requires_grad = False

# Replace final layer
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)

model = model.to(device)

# Loss & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)



In [ ]:
# Train
num_epochs = 8

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for inputs, labels in dataloaders['train']:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss:.4f}")

Epoch 1, Loss: 18.3171
Epoch 2, Loss: 11.0400
Epoch 3, Loss: 9.1587
Epoch 4, Loss: 8.7079
Epoch 5, Loss: 7.3463
Epoch 6, Loss: 7.7719
Epoch 7, Loss: 7.1451
Epoch 8, Loss: 6.8675


In [ ]:
# Evaluation
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for inputs, labels in dataloaders['val']:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))


Classification Report:

              precision    recall  f1-score   support

     chicken       0.97      0.75      0.85        52
        duck       0.89      0.99      0.94       109

    accuracy                           0.91       161
   macro avg       0.93      0.87      0.89       161
weighted avg       0.92      0.91      0.91       161



The model achieved an overall accuracy of 91%, indicating strong performance in distinguishing between chicken and duck images.

The precision for the chicken class is high (0.97), meaning the model makes very accurate predictions when identifying chickens. However, the recall is slightly lower (0.75), indicating that some chicken images are misclassified as ducks.

For the duck class, the model achieved a very high recall (0.99), meaning it correctly identifies almost all duck images.

This slight bias towards the duck class may be due to residual imbalance in the dataset, particularly in the validation set.

### Improved Model (Fine-Tuning + Data Augmentation)

To improve performance, we applied the following changes:
- Unfroze deeper layers (layer4) of the pre-trained model
- Added stronger data augmentation (ColorJitter)
- Increased number of training epochs
- Reduced learning rate for stable fine-tuning

In [ ]:
# Improved transforms
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ]),
}

In [ ]:
# Freeze all layers first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last block (key improvement)
for param in model.layer4.parameters():
    param.requires_grad = True

# Replace final layer
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)

model = model.to(device)

# Lower learning rate
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0003)

In [ ]:
# Train longer
num_epochs = 6

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for inputs, labels in dataloaders['train']:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss:.4f}")



Epoch 1, Loss: 1.0866
Epoch 2, Loss: 1.9779
Epoch 3, Loss: 1.4953
Epoch 4, Loss: 1.2158
Epoch 5, Loss: 0.4075
Epoch 6, Loss: 0.4853


In [ ]:
# Evaluation
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for inputs, labels in dataloaders['val']:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())

print("\nImproved Model - Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))


Improved Model - Classification Report:

              precision    recall  f1-score   support

     chicken       0.88      0.85      0.86        52
        duck       0.93      0.94      0.94       109

    accuracy                           0.91       161
   macro avg       0.90      0.90      0.90       161
weighted avg       0.91      0.91      0.91       161



### Final Model Performance and Analysis

After applying fine-tuning and data augmentation, the model was trained for a reduced number of epochs (6) to prevent overfitting.

The final model achieved an accuracy of **91%**, matching the initial model while also improving class balance. The recall for the chicken class improved from 0.75 to 0.85, indicating better detection of chicken images. At the same time, the model maintained strong performance on the duck class with a recall of 0.94.

This demonstrates that partial fine-tuning combined with appropriate training duration can improve model generalization without sacrificing overall accuracy. It also highlights the importance of avoiding overfitting by limiting the number of training epochs.

Thus, the final model provides both high accuracy and balanced performance across classes.
